In [49]:
import findspark
import pyspark

findspark.init()
sc = pyspark.SparkContext.getOrCreate()

In [50]:
moviesRDD = sc.textFile("data/movies.txt")
preferencesRDD = sc.textFile("data/preferences.txt")
watchedmoviesRDD = sc.textFile("data/watchedmovies.txt")

threshold = 2

In [51]:
moviesRDD.collect()

['movie1,Toy Story (1995),Animation',
 'movie2,Jumanji (1995),Adventure',
 'movie3,Grumpier Old Men (1995),Comedy',
 'movie4,Waiting to Exhale (1995),Comedy',
 'movie5,Father of the Bride Part II (1995),Comedy',
 'movie6,Heat (1995),Action',
 'movie7,Sabrina (1995),Comedy',
 'movie8,Tom and Huck (1995),Adventure',
 'movie9,Sudden Death (1995),Action',
 'movie10,GoldenEye (1995),Action']

In [52]:
preferencesRDD.collect()

['user1,Animation', 'user1,Comedy', 'user2,Action']

In [53]:
watchedmoviesRDD.collect()

['user1,movie1,201606061500,201606061650',
 'user1,movie3,201606061800,201606061834',
 'user1,movie4,201609061500,201609061605',
 'user1,movie5,201610061100,201610061450',
 'user2,movie6,201610081800,201610081845',
 'user2,movie3,201610091800,201610091834',
 'user2,movie4,201611051100,201611051105']

In [54]:
mappedMoviesRDD = moviesRDD.map(lambda x: (x.split(",")[0], x.split(",")[2]))
mappedMoviesRDD.collect()

[('movie1', 'Animation'),
 ('movie2', 'Adventure'),
 ('movie3', 'Comedy'),
 ('movie4', 'Comedy'),
 ('movie5', 'Comedy'),
 ('movie6', 'Action'),
 ('movie7', 'Comedy'),
 ('movie8', 'Adventure'),
 ('movie9', 'Action'),
 ('movie10', 'Action')]

In [55]:
mappedWatchedmoviesRDD = watchedmoviesRDD.map(lambda x: (x.split(",")[1], x.split(",")[0]))
mappedWatchedmoviesRDD.collect()

[('movie1', 'user1'),
 ('movie3', 'user1'),
 ('movie4', 'user1'),
 ('movie5', 'user1'),
 ('movie6', 'user2'),
 ('movie3', 'user2'),
 ('movie4', 'user2')]

In [56]:
joinedRDD = mappedWatchedmoviesRDD.join(mappedMoviesRDD)
joinedRDD.collect()

[('movie1', ('user1', 'Animation')),
 ('movie4', ('user1', 'Comedy')),
 ('movie4', ('user2', 'Comedy')),
 ('movie3', ('user1', 'Comedy')),
 ('movie3', ('user2', 'Comedy')),
 ('movie5', ('user1', 'Comedy')),
 ('movie6', ('user2', 'Action'))]

In [57]:
mappedJoinedRDD = joinedRDD.map(lambda x: (x[1][0], x[1][1]))
mappedJoinedRDD.collect()

[('user1', 'Animation'),
 ('user1', 'Comedy'),
 ('user2', 'Comedy'),
 ('user1', 'Comedy'),
 ('user2', 'Comedy'),
 ('user1', 'Comedy'),
 ('user2', 'Action')]

In [58]:
mappedPreferencesRDD = preferencesRDD.map(lambda x: (x.split(",")[0], x.split(",")[1]))
mappedPreferencesRDD.collect()

[('user1', 'Animation'), ('user1', 'Comedy'), ('user2', 'Action')]

In [59]:
cogroupedRDD = mappedJoinedRDD.cogroup(mappedPreferencesRDD)
cogroupedRDD.collect()

[('user2',
  (<pyspark.resultiterable.ResultIterable at 0x23fd1ace790>,
   <pyspark.resultiterable.ResultIterable at 0x23fd1d0a510>)),
 ('user1',
  (<pyspark.resultiterable.ResultIterable at 0x23fd28dd4d0>,
   <pyspark.resultiterable.ResultIterable at 0x23fd2801ed0>))]

In [60]:
def define_misleading_user(x):
    count = 0
    for genre in x[1][0]:
        if genre in x[1][1]:
            count += 1

    if count >= threshold:
        return x[0], 'Not Misleading'
    else:
        return x[0], 'Misleading User'

In [61]:
misleadingRDD = cogroupedRDD.map(lambda x: define_misleading_user(x))
misleadingRDD.collect()

[('user2', 'Misleading User'), ('user1', 'Not Misleading')]